# HYDROAWARE AFRICA project

## Preprocessing Water Body Components

### Notebook guide

This notebook prepares surface-water-body datasets required by DRYP. Use it after defining your model domain and before running simulations.

**What you will do:**
1. Read and inspect source datasets for lakes/reservoirs.
2. Convert or harmonize datasets to the DRYP grid.
3. Export model-ready layers and verify dimensions/projection consistency.

**Tip:** Keep all generated files in a dedicated output folder so they can be reused in later notebooks.


In [2]:
import rasterio
import numpy as np
from scipy.ndimage import label


def expand_region(array):
	"""
    Shrinks a binary region in a 2D array by removing 
    pixels around the regions.

    Parameters
	----------
    array:numpy array
		2D NumPy array of integers. 

    Returns
	-------
    	2D NumPy array with the shrunk region.

    This function first converts the input array to integers. 
    Then, it identifies and removes single-pixel protrusions 
    from the binary region represented by non-zero values 
    in the array. 
    """
	array = np.array(array, dtype=int)
	array[array < 0] = 0
	array[array > 0] = 1
	
	aux = array.copy()
	array[np.diff(aux, append=0, axis=1) == 1] = 1
	array[np.diff(aux, append=0, axis=0) == 1] = 1
	array[np.diff(aux, prepend=0, axis=0) == -1] = 1
	array[np.diff(aux, prepend=0, axis=1) == -1] = 1
	
	return array

def mask_lakes(array):
	array = np.array(array, dtype=int)
	array[array < 0] = 0
	array[array > 0] = 1
	return array

def get_water_body_parameters(depth, area=None):
    """This function calculates all water body parameters required to run
    the water body component
    
    Parameters
    ----------
    depth :	numpy array
        bathymetry of the water body [m]
    area :	numpy array
        surface area of the water body [m2]
    
    Returns
    -------
    tuple of lists/arrays
        name_lks : array
            labels of lake cells (for returned ids)
        ids_lks : list
            flat indices of all lake cells (flattened, grouped by lake label)
        size_lks : list
            number of cells in each lake
        ids_max_depth_lks : list
            flat indices of the cell with maximum depth in each lake
        depths : array
            depth values corresponding to ids_lks
    """
    # mask lakes from depth
    name_lks = (depth > 0).astype(int)
    #print(name_lks)
    # label lakes (connected components)
    labeled, num_features = label(name_lks)

    # build groups of flat indices per labeled lake (1..num_features)
    flat = labeled.flatten()
    ids_group_by_label = []
    for lab in range(1, num_features + 1):
        ids = list(np.where(flat == lab)[0])
        ids_group_by_label.append(ids)

    # flatten groups to single list of ids (grouped by label)
    ids_lks = [idx for grp in ids_group_by_label for idx in grp]

    # sizes per lake
    size_lks = [len(grp) for grp in ids_group_by_label]

    # find index of maximum depth within each group (global flat index)
    depth_flat = depth.flatten()
    ids_max_depth_lks = []
    for grp in ids_group_by_label:
        if len(grp) == 0:
            continue
        grp_depths = depth_flat[grp]
        imax = int(np.argmax(grp_depths))
        ids_max_depth_lks.append(grp[imax])

    # reduce name array to only the returned ids (labels for those ids)
    name_lks_out = flat[ids_lks]

    # reduce depth array to those ids
    depths_out = depth_flat[ids_lks]

    return name_lks_out, ids_lks, size_lks, ids_max_depth_lks, depths_out

In [3]:
def compute_lake_modified_dem(dem, bathymetry, riv_elevation):
    """
    Generates a modified DEM and river bottom elevation by expanding lakes,
    computing lake surface elevation, and blending with river elevations.
    Lake surface elevation uses the minimum elevation found along the border
    of the lake.

    This function can also be used when the domain contains multiple lakes.

    Parameters
    ----------
    dem : ndarray
        Original DEM raster.
    bathymetry : ndarray
        Water / mask input to detect lakes.
    riv_elevation : ndarray
        River bottom elevation raster.

    Returns
    -------
    dem_lks : ndarray
        Modified DEM including lake surfaces.
    river_bottom_mod : ndarray
        River bottom elevations modified at lake locations.
    """

    # ------------------------
    # 1. Lake mask
    # ------------------------
    mask = mask_lakes(bathymetry)

    # ------------------------
    # 2. Identify lake IDs
    # ------------------------
    lake_labels, lake_ids, *_ = get_water_body_parameters(bathymetry)

    # ------------------------
    # 3. Expand lake region
    # ------------------------
    mask_expand = expand_region(bathymetry)
    expanded_labels, expanded_ids, *_ = get_water_body_parameters(mask_expand)

    # ------------------------
    # 4. Create border elevations (expanded - lake interior)
    # ------------------------
    border = np.where((mask_expand > 0) & (mask == 0), dem, np.nan)

    # Flatten once for efficiency
    border_flat = border.ravel()

    lake_labels = np.asarray(lake_labels)
    expanded_labels = np.asarray(expanded_labels)
    lake_ids = np.asarray(lake_ids)
    expanded_ids = np.asarray(expanded_ids)

    # ------------------------
    # 5. Compute lake surface elevation using min border elevation
    # ------------------------
    surface = np.zeros(mask.size)  # flattened
    unique_lakes = np.unique(expanded_labels)

    for lake in unique_lakes:
        # expanded region pixel indices for this lake
        i_exp = np.where(expanded_labels == lake)[0]

        # find lake perimeter elevation
        min_elev = np.nanmin(border_flat[expanded_ids[i_exp]])

        # interior lake indices
        i_lake = np.where(lake_labels == lake)[0]
        surface[lake_ids[i_lake]] = min_elev

    surface = surface.reshape(mask.shape)

    # ------------------------
    # 6. Construct DEM with lake surface elevations
    # ------------------------
    dem_lks = dem.copy()
    dem_lks[mask > 0] = surface[mask > 0]

    # ------------------------
    # 7. Modify river bottom elevations inside lakes
    # ------------------------
    river_bottom_mod = np.where(mask > 0, dem_lks, riv_elevation)

    # ------------------------
    # 8. Average DEM and river bottom at expanded river locations
    # ------------------------
    avg_river_elevation = 0.5 * (river_bottom_mod + dem_lks)

    data_river_elevation = np.where(
        mask_expand > 0, avg_river_elevation, riv_elevation
    )

    return dem_lks, data_river_elevation

In [5]:
def save_masked_raster(masked_data, meta, out_path):
    meta.update({
        'dtype': 'float32',  # change dtype to float to accommodate NaN
        'nodata': np.nan     # set nodata value to NaN
    })
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(masked_data.astype('float32'), 1)  # write the masked data as float

In [6]:
path_dem0 = "/home/c1755103/AF/dataset/raw/hyd_af_dem_30s.tif"
path_dem = '/home/c1755103/AF/dataset/raw/hyd_af_dem_30s.tif' # this will be updated with the modified DEM after processing
path_bathymetry = '/home/c1755103/AF/dataset/postpp/af_bathymetry_30s.tif'
path_riv_len = '/home/c1755103/AF/dataset/postpp/af_river_length_30s_masked.tif'
path_river_bottom = '/home/c1755103/AF/dataset/postpp/af_dem_30s_q1.tif'

In [7]:
# ----------------------------
# Load rasters
# ----------------------------
with rasterio.open(path_dem0) as r1:
    dem = r1.read(1)
    dem_meta = r1.meta

with rasterio.open(path_dem) as r1:
    dem = r1.read(1)
    dem_meta = r1.meta

with rasterio.open(path_bathymetry) as r2:
    bathymetry = r2.read(1)
    bathymetry_meta = r2.meta

with rasterio.open(path_river_bottom) as r3:
    river_bottom = r3.read(1)
    river_bottom_meta = r3.meta

In [8]:
# ----------------------------
# Compute lake-modified DEM and river elevation
# ----------------------------
dem_lks, data_river_elevation = compute_lake_modified_dem(dem, bathymetry, river_bottom)

In [9]:
# ----------------------------
# Save outputs
# ----------------------------

path_dem_lks = '/home/c1755103/AF/dataset/postpp/af_dem_30s_lks.tif'
save_masked_raster(dem_lks, dem_meta, path_dem_lks)
print("Lake-modified DEM has been saved successfully.")
print(path_dem_lks)

path_river_elevation = '/home/c1755103/AF/dataset/postpp/af_river_elevation_30s_lks.tif'
save_masked_raster(data_river_elevation, river_bottom_meta, path_river_elevation)
print("River elevation raster has been saved successfully.")
print(path_river_elevation)

Lake-modified DEM has been saved successfully.
/home/c1755103/AF/dataset/postpp/af_dem_30s_lks.tif
River elevation raster has been saved successfully.
/home/c1755103/AF/dataset/postpp/af_river_elevation_30s_lks.tif


In [ ]:
# 1. Create mask of lakes
mask = mask_lakes(data2)

# 2. identify lakes ids
name_lks_out, ids_lks, _, _, _ = get_water_body_parameters(data2)

# 2. Expand lakes one cell
mask_expand = expand_region(data2)

# 2.1 Identify the minimum depth for the lake surface
#border = np.where(mask_expand-mask > 0, data0, border)
ename_lks_out, eids_lks, _, _, _ = get_water_body_parameters(mask_expand)

border = np.zeros_like(mask_expand)
border = np.where(mask_expand-mask > 0, data0, np.nan)

border_flat = border.flatten()
eids_lks = np.array(eids_lks)
ids_lks = np.array(ids_lks)

# 2.2 modify dem with new lake information
surface_lks = np.zeros_like(mask_expand.reshape(-1))
unique_ids = np.unique(ename_lks_out)
for ilake in unique_ids:
    iids_elks = np.where(ename_lks_out == ilake)[0]
    min_elevation = np.nanmin(border_flat[eids_lks[iids_elks]])
    iids_lks = np.where(name_lks_out == ilake)[0]
    surface_lks[ids_lks[list(iids_lks)]] = min_elevation

surface_lks = surface_lks.reshape(mask.shape)

dem_lks = data0.copy()
dem_lks[mask > 0] = surface_lks[mask > 0]

# 3. Make all river elevations at lakes equal to the surface elevation
river_bottom_mod = np.where(mask > 0, dem_lks, data3)
#river_bottom_mod = np.where(mask_expand > 0, dem_lks, data3)

# 4. get average of the two raster datasets
avg_river_elevation = (river_bottom_mod + dem_lks)*0.5

# 5. replace values in the expanded river bottom elevation
data_river_elevation = data3.copy()
data_river_elevation = np.where(data_expand > 0, avg_river_elevation, data3)